In [7]:
%matplotlib qt
import numpy as np
import tkinter as tk
from tkinter import filedialog
import os
import SimpleITK as sitk

import gyroid_utils
from gyroid_utils.utils import reload_all
reload_all()

working_path = os.getcwd()
print("Current working directory:", working_path)
parent_dir = os.path.dirname(working_path)

[gyroid_utils] version 3.0.6 loaded
gyroid_utils: all modules reloaded
Current working directory: c:\Users\cofo\Documents\02 - GitHub\GYROIDS\notebooks


In [8]:
# ---- convert tiff to mhd ----
root = tk.Tk()
input_file_path = filedialog.askdirectory() # this is the mhd file contnaing the whole image, no filter, no segmentation
print(input_file_path)
root.withdraw() 

gyroid_utils.CT_scans.convert_tiff_to_mhd(input_file_path, input_file_path, memory_saver=True)


T:/Micol/Cleaned_slices_FHH18_17
[INFO] gyroid_utils:((convert_tiff_to_mhd)): 974 TIFF images found.
[INFO] gyroid_utils:((convert_tiff_to_mhd)): CT scan of dimension (1130, 1130, 974) detected
[INFO] gyroid_utils:((convert_tiff_to_mhd)): Using default spacing: (0.2, 0.2, 0.2) and origin: (0.0, 0.0, 0.0)
Progress: [#############################-] 99%[INFO] gyroid_utils:((convert_tiff_to_mhd)): Saving as mhd...


In [9]:
#------  select the mhd file -------

# OPTION 1: Prompt the user to select a file using a file explorer

root = tk.Tk()
input_file_path = filedialog.askopenfilename() # this is the mhd file contnaing the whole image, no filter, no segmentation
print(input_file_path)
root.withdraw()

file_name = str(input_file_path).split('/')[-1].split('.')[0]


# open the file
working_path = os.path.dirname(input_file_path)
print(f"working path is {working_path}")
file_name = os.path.splitext(os.path.basename(input_file_path))[0]
print(f"file name is {file_name}")


image, spacing, origin = gyroid_utils.CT_scans.read_mhd_file(input_file_path)



T:/Micol/Cleaned_slices_FHH18_17.mhd
working path is T:/Micol
file name is Cleaned_slices_FHH18_17


2026-04-21 14:51:39,943 - CT_window_logger - INFO - starting window setup function


Clicked at coordinates: x=661, y=637, in slice 948, Pixel value: 253


2026-04-21 14:51:55,567 - CT_window_logger - INFO - window setup function succesfull


In [ ]:
# convert mhd to stl
#recreate x,y,z axis based on the spacing and origin
z = np.arange(image.shape[2]) * spacing[0]
y = np.arange(image.shape[1]) * spacing[1] 
x = np.arange(image.shape[0]) * spacing[2] 

Y, X, Z = np.meshgrid(y,x,z)
print(X.shape, Y.shape, Z.shape, image.shape)

verts, faces = gyroid_utils.mesh_tools.mesh_from_matrix(
    matrix= image,
    iso_level = 1,
    algo_step_size = 1,
    x = X,
    y = Y,
    z = Z,
    pad_width=10,
    pad_val=0,
    )

#verts, faces = gyroid_utils.mesh_tools.smooth_mesh(verts = verts, faces = faces, smoothing_factor=0.3, iterations=5)
faces, verts = gyroid_utils.mesh_tools.simplify_mesh(verts = verts, faces = faces, target=1000000, mode="fast")

gyroid_utils.viz.save_mesh_as_html(verts = verts, faces = faces, show_curvature_colorscale=True, file_name = str(input_file_path).split('.')[0])
gyroid_utils.mesh_tools.export_as_STL(verts, faces, path = str(input_file_path).split('.')[0] + '.stl')

(974, 1130, 1130) (974, 1130, 1130) (974, 1130, 1130) (974, 1130, 1130)
[INFO] gyroid_utils:((mesh_from_matrix)): Extracting isosurface mesh from matrix.
[INFO] gyroid_utils:((mesh_from_matrix)): Extracted mesh with 13761466 verts and 27524824 faces.
[INFO] gyroid_utils:((simplify_mesh)): Simplifying mesh: 27524824 faces → target 1000000
[INFO] gyroid_utils:((simplify_mesh)): Mesh simplification complete → 27392396 faces remain.
[INFO] gyroid_utils:((save_mesh_as_html)): Saving mesh visualization → 'T:/Micol/Cleaned_slices_FHH18_17.html'
[INFO] gyroid_utils:((save_mesh_as_html)): Reducing faces: 27392396 → 5000000 (centroid-based filtering)
[INFO] gyroid_utils:((save_mesh_as_html)): Curvature: processed vertex 0/13695252
[INFO] gyroid_utils:((save_mesh_as_html)): Curvature: processed vertex 10000/13695252
[INFO] gyroid_utils:((save_mesh_as_html)): Curvature: processed vertex 40000/13695252
[INFO] gyroid_utils:((save_mesh_as_html)): Curvature: processed vertex 70000/13695252
[INFO] gyro